# 05 — Error analysis (test period)

> This system predicts glucose values for research and analysis. It is not a medical device and must not be used to determine insulin doses or make treatment decisions.

Uses the one-time test predictions (`python -m src.holdout`); see `reports/test_report.md`. Do not use these results to change the model choice.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, db, train
settings, mcfg = config.settings(), config.models()
duck = db.duck(read_only=True)
q = lambda sql: duck.execute(sql).df()

In [ ]:
test = pd.read_parquet(config.path('data/predictions/test_predictions.parquet'))
rf = test[test.model == 'random_forest'].copy()
rf['err'] = rf.y_pred - rf.y_true
rf.groupby('horizon').err.agg(mae=lambda e: e.abs().mean(), bias='mean').round(3)

## Predicted vs actual (30 min)

In [ ]:
r30 = rf[rf.horizon == 30]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r30.y_true, r30.y_pred, s=2, alpha=0.2)
ax.plot([2, 22], [2, 22], color='k', lw=0.8)
for v in (3.9, 10.0):
    ax.axvline(v, color='grey', ls='--', lw=0.6); ax.axhline(v, color='grey', ls='--', lw=0.6)
ax.set_xlabel('actual (mmol/L)'); ax.set_ylabel('predicted (mmol/L)')

## Shrinkage toward the middle

Mean prediction error by actual glucose: positive at low values (predictions too high) and negative at high values. This is why point predictions miss lows.

In [ ]:
r30.groupby(pd.cut(r30.y_true, [0, 3.0, 3.9, 7, 10, 13.9, 30])).err.agg(['mean', 'count']).round(3)